# ECCO p02996 Job deadlines
#### Payload type: CPU-bound Python loop over every time unit up to the largest deadline

Source: [ECCO edit/test row 418](https://huggingface.co/datasets/CodeEff/ECCO/viewer/edit/test?row=418) - the slow submission, verbatim, wrapped as `solve(text)`.

In [ ]:
%load_ext jumper_extension
%perfmonitor_fast_setup

In [ ]:
import random
import zlib

In [ ]:
# Inputs scaled within the problem's constraints, fixed by seed
def generate_input(seed, jobs=200000, horizon=40000000):
    """N = 2 * 10^5 jobs with deadlines up to ~horizon (constraint: 10^9).

    Even seeds are schedulable; odd seeds get one job with deadline 1,
    so both answers occur.
    """
    rng = random.Random(seed)
    durations = [rng.randint(1, max(1, 2 * horizon // jobs)) for _ in range(jobs)]
    deadlines, total = [], 0
    for duration in durations:
        total += duration
        deadlines.append(total + rng.randint(0, horizon // 10))
    if seed % 2:
        deadlines[rng.randrange(jobs)] = 1
    pairs = list(zip(durations, deadlines))
    rng.shuffle(pairs)
    return f"{jobs}\n" + "\n".join(f"{a} {b}" for a, b in pairs) + "\n"


scaled_inputs = [generate_input(seed) for seed in range(4)]

In [ ]:
import heapq
#これを加える！！！！
def _heappush_max(heap, item):
    heap.append(item)
    heapq._siftdown_max(heap, 0, len(heap)-1)
def _heappop_max(heap):
    """Maxheap version of a heappop."""
    lastelt = heap.pop()    # raises appropriate IndexError if heap is empty
    if heap:
        returnitem = heap[0]
        heap[0] = lastelt
        heapq._siftup_max(heap, 0)
        return returnitem
    return lastelt


def solve(text):
    lines = iter(text.splitlines())
    input = lambda: next(lines)

    n = int(eval(input()))
    works = [[]] * n
    for i in range(n):
      a, b = list(map(int, input().split(' ')))
      works[i] = [a, b]

    ans = 'Yes'
    sorted_works = sorted(works, key=lambda x: x[1], reverse=True)
    max_deadline = sorted_works[0][1]
    # print('max_deadline', max_deadline)
    h = []
    deadline_i = 0
    work_day = max_deadline # 作業日
    # 締め切り最終日から0日に向かってループ
    for d in range(max_deadline, -1, -1):
      # その日が締め切りの作業時間を優先度キューに追加
      for i in range(deadline_i, n):
        work_time, deadline = sorted_works[i]
        if deadline == d:
          _heappush_max(h, work_time)
          deadline_i += 1
        else:
          break
      # 作業ができない日であれば、次のループへ
      if d > work_day:
        continue

      # 作業ができる日

      # 作業がなければ次のループへ
      if len(h) == 0:
        continue


      # その日に終了しても良い作業の中で時間が最長の作業時間を取得
      max_work_time = _heappop_max(h)
      # 次の作業日を設定
      work_day = d - max_work_time

      # 0日より前になる = 作業量が多いので実行不可
      if work_day < 0:
        ans = 'No'



    return ans
    # print(works)
    # print(sorted_works)


answers = [solve(text) for text in scaled_inputs]
checksum = sum(zlib.crc32(answer.encode()) for answer in answers)

In [ ]:
%perfmonitor_ai_review --benchmark --replay-mode full